# BOOMI Dataset Exploration

Task 2.1 — load BOOMI, inspect splits, inspect real samples, parse `spec`, inspect L0/L1/L2/L3 examples, verify available fields.

In [ ]:
from datasets import load_dataset

ds = load_dataset("BDivyesh/boomi-stage-a-text-spec")
ds

## Findings

- **Splits**: train=59,564 rows / validation=4,000 / test=4,000. Rows = unique plans × 4 levels.
- **Unique plans**: train=14,891, validation=1,000, test=1,000 — **no `plan_id` overlap between splits** (verified directly), so the leakage rule from Task 7.5 already holds for the raw data.
- **Fields**: `plan_id`, `level` (`L0`-`L3`), `caption` (str), `spec` (JSON string).
- **`spec` is identical across L0-L3 for a given `plan_id`** — only `caption` changes, growing from terse structured facts (L0) to full prose (L3). This is a caption-detail ladder over a fixed target spec, not 4 different plans.
- **`spec` schema**: `adjacency` (`a_type`, `b_type`, `via`), `bhk_label`, `plot` (`area_m2`, `width_mm`, `depth_mm`), `room_program` (`type`, `count`, `target_area_m2`), `total_area_m2`. No coordinates/geometry (consistent with the "stage-a-text-spec" name).
- **Room types (8)**: BATH, LIVING, BED ROOM, KITCHEN, BALCONY, STORE, STAIR, PARKING.
- **Relationship types (`via`, 3)**: adjacent, door, window.
- **BHK labels**: 1BHK-7BHK plus STUDIO; 2BHK and 3BHK dominate.
- **Quality**: 0 invalid JSON, 0 missing required fields, 0 unknown room types, 0 impossible dimensions, 0 duplicate `(plan_id, level)` records. 324 room-level area outliers (<1 m² or >200 m², mostly PARKING/large LIVING) worth reviewing before training.

Full numbers: [`data/reports/boomi_statistics.json`](../data/reports/boomi_statistics.json), [`data/reports/boomi_quality_report.json`](../data/reports/boomi_quality_report.json).